# Discussion Section Week 3 - 4/17/25

## Before you Start Download this File

Make sure to download this CSV to use during this lab.  
[processed_pcap.csv](https://drive.google.com/file/d/1jJs-bG4-zdUTSwzLb3imf39WGPzrAzns/view?usp=sharing)   

## Converting a PCAP to CSV

The CSV you downloaded was generated from a `.pcap` capture file using `tshark`, the command-line version of Wireshark. You can reproduce it yourself with the following command:

```bash
tshark -r capture.pcap -T fields \
  -e frame.number \
  -e frame.time_relative \
  -e ip.src \
  -e ip.dst \
  -e tcp.srcport \
  -e tcp.dstport \
  -e ip.proto \
  -e tcp.seq \
  -e tcp.ack \
  -e tcp.flags \
  -e tcp.len \
  -e tcp.window_size \
  -E header=y -E separator=, \
  > processed_pcap.csv
```

Each `-e` flag extracts a specific field from every packet. Here's what each column represents:

| CSV Column | tshark Field | Description |
|---|---|---|
| `frame_no` | `frame.number` | Sequential packet number in the capture |
| `time` | `frame.time_relative` | Time (in seconds) since the first packet |
| `src_ip` | `ip.src` | Source IP address |
| `dst_ip` | `ip.dst` | Destination IP address |
| `src_port` | `tcp.srcport` | Source TCP port |
| `dst_port` | `tcp.dstport` | Destination TCP port |
| `protocol` | `ip.proto` | IP protocol number (e.g., `6` for TCP, `17` for UDP) |
| `seq` | `tcp.seq` | TCP sequence number |
| `ack` | `tcp.ack` | TCP acknowledgment number |
| `flags` | `tcp.flags` | TCP flags as a hex value (e.g., `0x0012` for SYN-ACK) |
| `payload_len` | `tcp.len` | Length of TCP payload in bytes |
| `window_size` | `tcp.window_size` | TCP receive window size |

This gives us a structured, tabular representation of the packet capture that we can easily load into Pandas.

# Analyzing Network Data with Pandas

## Why Pandas?

[Pandas](https://pandas.pydata.org/) is a powerful Python library for data analysis and manipulation. It offers several key benefits for working with network data:

- **Tabular data handling** — Network logs, packet captures, and flow records naturally fit into table-like structures. Pandas DataFrames make it easy to load, inspect, and manipulate these datasets.
- **Efficient filtering and aggregation** — Quickly filter packets by protocol, IP address, or port, and compute summary statistics like throughput, latency distributions, and packet counts.
- **Built-in I/O support** — Read and write CSV, JSON, Parquet, and other formats with a single function call, making it straightforward to work with data exported from tools like Wireshark, tcpdump, or NetFlow collectors.
- **Integration with visualization libraries** — Pandas integrates seamlessly with Matplotlib, Seaborn, and Plotly for creating plots and charts to visualize network behavior.
- **Vectorized operations** — Perform computations across entire columns without writing explicit loops, leading to cleaner and faster code.

## About This Lab

In this lab, we will introduce you to using Pandas for analyzing network data programmatically. You will learn how to load network datasets, explore their structure, filter and transform the data, and extract meaningful insights — all using Python and Pandas.


## Getting Started

### Inspecting the Columns

Before diving into analysis, it's good practice to verify that the CSV columns match what we expect from the `tshark` command above. We can use `df.columns` to list all column names, and `df.dtypes` to see the data types Pandas inferred for each column.

In [ ]:
import pandas as pd

df = pd.read_csv("processed_pcap.csv")

# Keep only the columns we use in this lab (ignore extra columns if present)
desired_cols = [
    "frame.number",
    "frame.time_relative",
    "ip.src",
    "ip.dst",
    "tcp.srcport",
    "tcp.dstport",
    "ip.proto",
    "tcp.seq",
    "tcp.ack",
    "tcp.flags",
    "tcp.len",
    "tcp.window_size",
]
df = df[[c for c in desired_cols if c in df.columns]]

df.head()

In [ ]:
print("Columns in CSV:")
print(df.columns.tolist())
print(f"\nNumber of columns: {len(df.columns)}")

In [ ]:
# Check data types and basic stats for each column
df.dtypes

Notice that Pandas automatically infers data types when loading a CSV. Numeric fields like `frame.number`, `tcp.len`, and `tcp.window_size` are read as integers, while `frame.time_relative` is read as a float. Fields like `ip.src`, `ip.dst`, and `tcp.flags` are stored as strings (`object` type) since they contain non-numeric characters.

Understanding the data types helps you know which operations are available — for example, you can compute the mean of numeric columns directly, but string columns require different handling.

## Exercise A

**Goal:** Identify the TCP flows that carried the most payload bytes and the most data packets.

### Step 1: Filter for TCP packets only

Our capture may contain packets from multiple protocols. The IP header's `ip.proto` field indicates which transport protocol a packet belongs to.

In [ ]:
# Filter for TCP packets






### Step 2: Filter out pure ACKs (zero-payload packets)

Not every TCP packet carries application data. Pure acknowledgements (ACKs) have a payload length of zero — they exist only to confirm receipt of data. Since we want to measure actual data transfer, we need to filter these out.

In [ ]:
# Keep only data packets (non-zero payload)






### Step 3: Find the flow sending the most traffic (number of packets and bytes)

We can identify a (directional) TCP flow using the 5-tuple: `ip.src`, `ip.dst`, `tcp.srcport`, `tcp.dstport`, and `ip.proto`. We group by these fields using `groupby()` and then aggregate to compute:
- **Total payload bytes** — sum of `tcp.len`
- **Total data packets** — count of rows
- **Flow duration** — difference between the last and first `frame.time_relative` timestamp

In [ ]:
# Group by 5-tuple and compute stats per flow









## Exercise B

**Goal:** Identify the set of *data packet retransmissions* for each flow.

A simple way to detect retransmissions is to look for **reused TCP sequence numbers** within the same (directional) flow. If we restrict to **data packets** (`tcp.len > 0`) and then see the same `tcp.seq` value appear multiple times for the same 5‑tuple flow, that strongly suggests a retransmission (or, occasionally, capture duplication/out-of-order effects).

In [ ]:
# Your CODE here









## Exercise C

**Goal:** Rolling throughput analysis using windowing.

We’ll estimate throughput by (1) binning data packets into fixed-size time bins, and (2) taking a rolling sum over a sliding window of bins. The rolling window smooths out burstiness so you can see trends over time.

In [ ]:
# Your CODE here









## Exercise D

**Goal:** RTT approximation by matching ACKs with their sequence numbers.

For a TCP data packet with sequence number `tcp.seq` and payload length `tcp.len`, the receiver will (eventually) ACK the next expected byte: `tcp.ack = tcp.seq + tcp.len`.

We can approximate RTT by joining:
- **data packets** in one direction, with
- **ACK packets** in the reverse direction

…on that expected ACK value, and then computing $\text{RTT} = t_{ack} - t_{sent}$ for the first matching ACK.

In [ ]:
# Your CODE here









## Exercise E

**Goal:** Track download progress over time by computing how long it takes to reach each 10% milestone (10%, 20%, …, 100%) of the total bytes downloaded by the client.


In [ ]:
# Your CODE here









### Submission Instructions

To receive full credit for this lab, you must submit your completed notebook as a **participation check** on [Gradescope](https://www.gradescope.com) - add code is **4DWRD5**.

> **Due Date**: Make sure to submit by the end of the weekend (Sunday 4/19) @ 11:59PM